# Предсказание правильности ответа — обучение модели

Датасет **MathE**: ответы студентов на математические задачи. По теме, уровню сложности и стране студента модель предсказывает, ответит ли он **верно** (1) или **неверно** (0).

Запускай ячейки по порядку сверху вниз.

## Шаг 1. Загружаем данные
Выбери файл `math_e.csv`.

In [ ]:
from google.colab import files
uploaded = files.upload()   # выбери math_e.csv

## Шаг 2. Читаем таблицу
У этого файла разделитель `;` и кодировка `latin-1` — указываем их явно.

In [ ]:
import pandas as pd

df = pd.read_csv("math_e.csv", sep=";", encoding="latin-1", on_bad_lines="skip")

print("Размер:", df.shape)
print("Колонки:", df.columns.tolist())
df.head()

## Шаг 3. Выбираем признаки и цель
- **Цель:** `Type of Answer` (0 — неверно, 1 — верно).
- **Признаки:** страна, уровень, тема, подтема.

In [ ]:
features = ["Student Country", "Question Level", "Topic", "Subtopic"]
target = "Type of Answer"

X = df[features].fillna("Unknown")
y = df[target]

print("Примеров:", len(X))
print("Баланс классов (0=неверно, 1=верно):")
print(y.value_counts(normalize=True).round(3).to_dict())

## Шаг 4. Делим на train/test

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Обучение:", len(X_train), "| Тест:", len(X_test))

## Шаг 5. Baseline — «отметка на стене»
Всегда предсказывает самый частый класс. Моя модель обязана быть лучше.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
bp = baseline.predict(X_test)

print("BASELINE")
print("  Accuracy:", round(accuracy_score(y_test, bp), 3))
print("  macro-F1:", round(f1_score(y_test, bp, average="macro"), 3))

## Шаг 6. Моя модель (Pipeline: OneHotEncoder + LogisticRegression)
Категории превращаем в числа через OneHotEncoder, дальше — логистическая регрессия. Всё в одном Pipeline, обучается только на train (без утечки).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

model = Pipeline([
    ("prep", ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), features)
    ])),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
]).fit(X_train, y_train)

pred = model.predict(X_test)

print("МОЯ МОДЕЛЬ")
print("  Accuracy:", round(accuracy_score(y_test, pred), 3))
print("  macro-F1:", round(f1_score(y_test, pred, average="macro"), 3))
print("\n(эти числа впиши в таблицу метрик в README)")
print("\n", classification_report(y_test, pred, zero_division=0))

## Шаг 7. Сохраняем модель и скачиваем
Файл `mathe_model.pkl` заливаем в Hugging Face Space рядом с `app.py`.

In [ ]:
import joblib
from google.colab import files

joblib.dump(model, "mathe_model.pkl")
files.download("mathe_model.pkl")
print("Готово — скачай mathe_model.pkl и залей в Space.")